# Evaluating a Local LLM for Safe Warehouse Robot Planning
## Mac and local Jupyter edition

This notebook runs the complete dissertation experiment on a Mac. It does not use Google Colab and it does not upload any folders. Keep this notebook inside the extracted `warehouse-llm-dissertation` folder and run each cell in order.

The project evaluates Qwen on direction reasoning, collision-free route planning, and pick-and-deliver task planning in a simulated warehouse. It also contains an exploratory hybrid LLM + A* comparison.

## 1. Find and install the project

This cell automatically finds the project folder, including folders such as `warehouse-llm-dissertation-2` in Downloads. It installs all required Python packages into the Python environment used by this Jupyter kernel.

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

if platform.system() != 'Darwin':
    raise RuntimeError('This notebook is the Mac edition and requires macOS.')

def find_project_folder():
    candidates = [Path.cwd(), Path.cwd().parent]
    downloads = Path.home() / 'Downloads'
    if downloads.exists():
        downloaded_projects = sorted(
            downloads.glob('warehouse-llm-dissertation*'),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
        candidates.extend(path for path in downloaded_projects if path.is_dir())
    for candidate in candidates:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'warehouse_llm').is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        'Project folder not found. Extract the ZIP in Downloads and open this notebook from that folder.'
    )

PROJECT_DIR = find_project_folder()
os.chdir(PROJECT_DIR)
print('Mac:', platform.mac_ver()[0])
print('Python:', sys.version.split()[0])
print('Project folder:', PROJECT_DIR)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--editable', '.[dev]'],
    cwd=PROJECT_DIR,
    check=True,
)
print('Project installed successfully.')

## 2. Test the simulator

The first command runs the automated software tests. The second runs the ground-truth oracle through all three experiment pipelines. The oracle is a pipeline check, not an LLM result.

In [ ]:
def run_python(*arguments):
    return subprocess.run(
        [sys.executable, *arguments],
        cwd=PROJECT_DIR,
        check=True,
    )

run_python('-m', 'pytest', '-q')
run_python(
    '-m', 'warehouse_llm.cli', 'all',
    '--client', 'oracle',
    '--cases', '2',
    '--output-dir', 'results/oracle_check',
)
print('Simulator and experiment pipeline verified.')

## 3. Start Ollama and download Qwen

Install Ollama for macOS from https://ollama.com/download/mac before running this cell. The cell opens the Ollama application, waits for its local server, and downloads `qwen:7b` if necessary. The model download is approximately 4.5 GB and can take several minutes.

In [ ]:
import json
import shutil
import time
import urllib.error
import urllib.request

OLLAMA_URL = 'http://127.0.0.1:11434'
MODEL_NAME = 'qwen:7b'
OLLAMA_TIMEOUT_SECONDS = 900
OLLAMA_MAX_TOKENS = 512

def get_local_models():
    try:
        with urllib.request.urlopen(f'{OLLAMA_URL}/api/tags', timeout=3) as response:
            return json.loads(response.read().decode('utf-8')).get('models', [])
    except (urllib.error.URLError, TimeoutError, json.JSONDecodeError):
        return None

models = get_local_models()
if models is None:
    app_check = subprocess.run(
        ['open', '-Ra', 'Ollama'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    ollama_cli = shutil.which('ollama')
    if app_check.returncode == 0:
        print('Starting the Ollama Mac application...')
        subprocess.run(['open', '-a', 'Ollama'], check=True)
    elif ollama_cli:
        print('Starting the Ollama command-line server...')
        ollama_server_process = subprocess.Popen(
            [ollama_cli, 'serve'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    else:
        raise RuntimeError(
            'Ollama is not installed. Download it from https://ollama.com/download/mac, move it to Applications, open it once, then rerun this cell.'
        )
    for _ in range(30):
        time.sleep(2)
        models = get_local_models()
        if models is not None:
            break

if models is None:
    raise RuntimeError('Ollama was found but its local server did not start. Open Ollama manually and rerun this cell.')

installed_names = {model.get('model', model.get('name', '')) for model in models}
if MODEL_NAME not in installed_names:
    print(f'Downloading {MODEL_NAME}. Keep this notebook open...')
    request = urllib.request.Request(
        f'{OLLAMA_URL}/api/pull',
        data=json.dumps({'model': MODEL_NAME, 'stream': True}).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urllib.request.urlopen(request, timeout=7200) as response:
        last_percentage = -1
        for raw_line in response:
            update = json.loads(raw_line.decode('utf-8'))
            total = update.get('total', 0)
            completed = update.get('completed', 0)
            if total:
                percentage = int(completed * 100 / total)
                if percentage >= last_percentage + 10:
                    print(f'Download progress: {percentage}%')
                    last_percentage = percentage
            elif update.get('status'):
                print(update['status'])
else:
    print(f'{MODEL_NAME} is already installed.')

print('Ollama and Qwen are ready.')

## 4. Check one response from the real LLM

This confirms that Python can communicate with Qwen through Ollama.

In [ ]:
from warehouse_llm.llm import OllamaClient

qwen = OllamaClient(
    model=MODEL_NAME,
    base_url=OLLAMA_URL,
    timeout=OLLAMA_TIMEOUT_SECONDS,
    max_tokens=OLLAMA_MAX_TOKENS,
    temperature=0.0,
)
test_response = qwen.generate('Reply with exactly one word: READY')
print('Model:', test_response.model)
print('Response:', test_response.text)
print('Latency in milliseconds:', round(test_response.latency_ms, 1))

## 5. Run the small pilot experiment

The pilot creates 16 direction cases, 12 route cases and 12 pick-and-deliver cases. Inspect the outputs before running the larger final experiment.

In [ ]:
def run_experiment(task, cases, seed, output_directory):
    run_python(
        '-m', 'warehouse_llm.cli', task,
        '--client', 'ollama',
        '--model', MODEL_NAME,
        '--timeout', str(OLLAMA_TIMEOUT_SECONDS),
        '--max-tokens', str(OLLAMA_MAX_TOKENS),
        '--cases', str(cases),
        '--seed', str(seed),
        '--output-dir', output_directory,
    )

run_experiment('direction', 1, 202600, 'results/pilot')
run_experiment('route', 1, 202700, 'results/pilot')
run_experiment('mission', 1, 202800, 'results/pilot')
print('Pilot experiment finished.')

In [ ]:
import pandas as pd
from IPython.display import display

for csv_file in sorted((PROJECT_DIR / 'results' / 'pilot').glob('*.csv')):
    print('\nFile:', csv_file.name)
    display(pd.read_csv(csv_file).head())

## 6. Run the final experiment

Only change `RUN_FINAL_EXPERIMENT` to `True` after the pilot outputs have been checked. The complete experiment can take several hours on a Mac because it requests 960 LLM responses. Keep the charger connected and prevent the Mac from sleeping. Do not change the seeds after viewing the results.

In [ ]:
RUN_FINAL_EXPERIMENT = False

DIRECTION_CASES_PER_CONDITION = 30
ROUTE_CASES_PER_CONDITION = 20
MISSION_CASES_PER_CONDITION = 20

FINAL_OUTPUTS = {
    'direction': PROJECT_DIR / 'results/final/direction_qwen-7b.csv',
    'route': PROJECT_DIR / 'results/final/route_qwen-7b.csv',
    'mission': PROJECT_DIR / 'results/final/mission_qwen-7b.csv',
}

def run_final_unless_complete(task, cases, seed):
    if FINAL_OUTPUTS[task].exists():
        print(f'Skipping {task}: completed CSV already exists.')
        return
    print(f'Starting final {task} experiment...')
    run_experiment(task, cases, seed, 'results/final')

if RUN_FINAL_EXPERIMENT:
    keep_awake_process = subprocess.Popen(
        ['caffeinate', '-dimsu'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    try:
        run_final_unless_complete('direction', DIRECTION_CASES_PER_CONDITION, 202600)
        run_final_unless_complete('route', ROUTE_CASES_PER_CONDITION, 202700)
        run_final_unless_complete('mission', MISSION_CASES_PER_CONDITION, 202800)
        print('Final experiment finished.')
    finally:
        keep_awake_process.terminate()
else:
    print('Final experiment not started.')
    print('Check the pilot first, then change RUN_FINAL_EXPERIMENT to True and rerun this cell.')

## 7. Run the exploratory hybrid LLM + A* experiment

This extension does not replace the original experiment. Qwen reads the same 240 mission instructions and returns only the action, pallet ID and dock ID. The verified A* planner then generates the route. This provides a paired comparison between direct LLM planning and the hybrid architecture. Run the 12-case pilot first, inspect it, and then run the final hybrid experiment.

In [ ]:
import pandas as pd
from IPython.display import display

def run_hybrid_experiment(cases, output_directory):
    run_python(
        '-m', 'warehouse_llm.cli', 'hybrid',
        '--client', 'ollama',
        '--model', MODEL_NAME,
        '--timeout', str(OLLAMA_TIMEOUT_SECONDS),
        '--max-tokens', str(OLLAMA_MAX_TOKENS),
        '--cases', str(cases),
        '--seed', '202800',
        '--output-dir', output_directory,
    )

RUN_HYBRID_PILOT = False

if RUN_HYBRID_PILOT:
    run_hybrid_experiment(1, 'results/hybrid_pilot')
    hybrid_pilot = pd.read_csv(PROJECT_DIR / 'results/hybrid_pilot/hybrid_qwen-7b.csv')
    display(hybrid_pilot.head(12))
    print('Hybrid pilot valid-mission rate:', hybrid_pilot['valid_hybrid_mission'].mean())
else:
    print('Change RUN_HYBRID_PILOT to True and rerun this cell.')

In [ ]:
RUN_HYBRID_EXPERIMENT = False
HYBRID_CASES_PER_CONDITION = 20
HYBRID_OUTPUT = PROJECT_DIR / 'results/hybrid/hybrid_qwen-7b.csv'

if RUN_HYBRID_EXPERIMENT:
    if HYBRID_OUTPUT.exists():
        print('Skipping hybrid experiment: completed CSV already exists.')
    else:
        keep_awake_process = subprocess.Popen(
            ['caffeinate', '-dimsu'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        try:
            run_hybrid_experiment(HYBRID_CASES_PER_CONDITION, 'results/hybrid')
            print('Hybrid experiment finished successfully.')
        finally:
            keep_awake_process.terminate()
else:
    print('Hybrid experiment not started.')
    print('Check the hybrid pilot, then change RUN_HYBRID_EXPERIMENT to True.')

## 8. Analyse the final and hybrid results

Run the next cell only after the final experiment finishes. It calculates the actual accuracy from the CSV files, checks that all 960 cases are present without duplicate scenario IDs, and generates confidence intervals, failure counts, graphs and a brief dissertation-ready report.

In [ ]:
final_csv_files = sorted((PROJECT_DIR / 'results' / 'final').glob('*.csv'))
if not final_csv_files:
    print('No final CSV files found. Run Section 6 first.')
else:
    analysis_arguments = [
        'analyse_results.py',
        '--input-dir', 'results/final',
        '--output-dir', 'results/analysis',
    ]
    if sorted((PROJECT_DIR / 'results' / 'hybrid').glob('*.csv')):
        analysis_arguments.extend(['--hybrid-dir', 'results/hybrid'])
    run_python(*analysis_arguments)
    print('Analysis complete. Files are in results/analysis.')

    import pandas as pd
    from IPython.display import Markdown, display

    analysis_dir = PROJECT_DIR / 'results' / 'analysis'
    print('\nDATA-INTEGRITY CHECKS')
    display(pd.read_csv(analysis_dir / 'experiment_integrity.csv'))
    print('\nOVERALL ACCURACY TABLE')
    display(pd.read_csv(analysis_dir / 'overall_accuracy_summary.csv'))
    print('\nBRIEF REPORT')
    display(Markdown((analysis_dir / 'brief_accuracy_report.md').read_text(encoding='utf-8')))

In [ ]:
from IPython.display import Image, display

for image_file in sorted((PROJECT_DIR / 'results' / 'analysis').glob('*.png')):
    print(image_file.name)
    display(Image(filename=str(image_file)))

## 9. Export the result evidence

First save this notebook with **Command + S**. Then run the cell below. The results ZIP will be created inside the project folder.

In [ ]:
import shutil

results_folder = PROJECT_DIR / 'results'
archive_base = PROJECT_DIR / 'warehouse_llm_results_mac'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=results_folder)
print('Results ZIP created at:')
print(archive_path)
print('Keep this ZIP together with the saved notebook and dissertation.')